# NB-06-01 — Data Preparation Workbench

**Module:** M06 · **Time:** 60–90 minutes · **Prerequisites:** L06-01, L06-02

Profile a dataset, impute, encode, scale, clip outliers and split — in that order, with the
preprocessing fitted on training rows only. By the end you will have a reusable
`Pipeline` and a measured before/after for outlier handling.

## 1. Setup

In [1]:
import sys
from pathlib import Path

for parent in [Path.cwd(), *Path.cwd().parents]:
    candidate = parent / "modules" / "M06-machine-learning-fundamentals" / "code"
    if candidate.is_dir():
        sys.path.insert(0, str(candidate))
        break
print("code directory:", candidate)

code directory: C:\Users\Nadeem\Documents\Artificial-Intelligence-Course\modules\M06-machine-learning-fundamentals\code


In [2]:
import numpy as np
import pandas as pd
import datasets as ds
from prep import (describe_frame, SimpleImputer, OneHotEncoder, StandardScaler,
                  MinMaxScaler, Pipeline, train_test_split, train_val_test_split,
                  clip_outliers_iqr, correlation_with_target)

## 2. Profile before modelling

In [3]:
students = ds.students()
print(describe_frame(students).to_string(index=False))
print()
print("pass rate:", round(students["passed"].mean(), 3),
      "| missing cells:", int(students.isna().sum().sum()))

            column   dtype  missing  unique    min    max
       study_hours float64        0     469    0.0  21.26
    attendance_pct float64       21     286   45.6  100.0
       prior_grade float64        0     369   18.4  100.0
       sleep_hours float64       14     329   3.81  10.19
     part_time_job     str        0       2      -      -
parental_education     str        0       3      -      -
  internet_at_home     str        0       2      -      -
       final_score float64        0     552  34.58  84.51
            passed   int64        0       2      0      1

pass rate: 0.728 | missing cells: 35


Read the `dtype` column: three categorical columns need encoding, and two numeric
columns have missing values. `final_score` is the target's parent — it must not be a feature
when predicting `passed`.

## 3. Split first

In [4]:
cols = ["study_hours", "attendance_pct", "prior_grade", "sleep_hours"]
X = students[cols].to_numpy()
y = students["final_score"].to_numpy()
X_train, X_test, y_train, y_test = train_test_split(X, y, 0.25, seed=0)
print("train/test rows:", len(X_train), len(X_test))
print("NaNs in train:", int(np.isnan(X_train).sum()))

train/test rows: 450 150
NaNs in train: 22


## 4. Impute, scale, encode — fitted on train

In [5]:
imputer = SimpleImputer("median").fit(X_train)
X_train_i = imputer.transform(X_train)
print("NaNs after imputation:", int(np.isnan(X_train_i).sum()))

scaler = StandardScaler().fit(X_train_i)
scaled = scaler.transform(X_train_i)
print("means:", scaled.mean(axis=0).round(6))
print("stds :", scaled.std(axis=0).round(3))

encoder = OneHotEncoder().fit(students[["parental_education"]].iloc[:450])
print("columns:", encoder.get_feature_names())
print("shape:", encoder.transform(students[["parental_education"]]).shape)

NaNs after imputation: 0
means: [ 0.  0. -0.  0.]
stds : [1. 1. 1. 1.]
columns: ['col0_none', 'col0_school', 'col0_university']
shape: (600, 3)


Now apply the same fitted objects to the test set. Note that you `transform` there,
never `fit` — that is the entire discipline.

In [6]:
X_test_i = imputer.transform(X_test)
print("test NaNs after imputation:", int(np.isnan(X_test_i).sum()))
print("test means after scaling :", scaler.transform(X_test_i).mean(axis=0).round(3))

test NaNs after imputation: 0
test means after scaling : [ 0.051 -0.085  0.173  0.148]


## 5. Outliers: measure the effect

In [7]:
houses = ds.houses()
print("max living area:", houses["living_area_sqft"].max())
print("price quantiles:", houses["price_thousands"].quantile([0.25, 0.5, 0.75]).round(1).to_dict())

clipped_price, lo, hi = clip_outliers_iqr(houses["price_thousands"])
clipped_area, alo, ahi = clip_outliers_iqr(houses["living_area_sqft"])
print("price fence:", round(lo, 1), round(hi, 1), "| rows changed:",
      int((clipped_price != houses["price_thousands"].to_numpy()).sum()))
print("area fence :", round(alo, 1), round(ahi, 1))

max living area: 12000.0
price quantiles: {0.25: 299.6, 0.5: 346.4, 0.75: 395.8}
price fence: 155.4 540.1 | rows changed: 7
area fence : 381.0 3207.0


The fence removes the impossible 12,000 sqft value and pulls in six luxury prices.
Lesson 2 measured the consequence: test R² rises from about 0.38 to 0.75. Do the fit here
to see it for yourself.

In [8]:
import linreg as lr

hcols = ["living_area_sqft", "bedrooms", "bathrooms", "age_years",
         "distance_to_centre_km", "garage"]
Xh, yh = houses[hcols].to_numpy(), houses["price_thousands"].to_numpy()
Xtr, Xte, ytr, yte = train_test_split(Xh, yh, 0.25, seed=0)
raw = lr.LinearRegression().fit(Xtr, ytr)
print("raw RMSE:", round(lr.rmse(yte, raw.predict(Xte)), 1),
      "R2:", round(raw.score(Xte, yte), 3))

houses["price_thousands"] = clipped_price
houses["living_area_sqft"] = clipped_area
Xh, yh = houses[hcols].to_numpy(), houses["price_thousands"].to_numpy()
Xtr, Xte, ytr, yte = train_test_split(Xh, yh, 0.25, seed=0)
clean = lr.LinearRegression().fit(Xtr, ytr)
print("clean RMSE:", round(lr.rmse(yte, clean.predict(Xte)), 1),
      "R2:", round(clean.score(Xte, yte), 3))

raw RMSE: 74.1 R2: 0.375
clean RMSE: 35.3 R2: 0.753


## 6. Feature selection is supervised

In [9]:
table = correlation_with_target(students[cols], students["final_score"])
print(table.round(3).to_string(index=False))

 feature  correlation
       0        0.555
       2        0.416
       1          NaN
       3          NaN


`attendance_pct` has missing values, so its correlation is computed on the rows where it
is present. If you selected features on **all** rows before splitting, the target's distribution
would inform your feature set — leakage, mild but real.

## 7. A pipeline makes leakage structural

In [10]:
from classifiers import LogisticRegressionGD
import evaluation as ev

pipe = Pipeline([
    ("impute", SimpleImputer("median")),
    ("scale", StandardScaler()),
    ("model", LogisticRegressionGD(lr=0.5, epochs=400)),
])
y_pass = students["passed"].to_numpy()
Xp, yp = students[cols].to_numpy(), y_pass
Xtr, Xte, ytr, yte = train_test_split(Xp, yp, 0.25, seed=0, stratify=yp)
pipe.fit(Xtr, ytr)
print("test accuracy:", round(pipe.score(Xte, yte), 3))
print("5-fold CV:", np.round(ev.cross_val_score(
    lambda: Pipeline([("impute", SimpleImputer("median")),
                      ("scale", StandardScaler()),
                      ("model", LogisticRegressionGD(lr=0.5, epochs=400))]),
    Xp, yp, k=5, stratified=True), 3))

test accuracy: 0.833


5-fold CV: [0.81  0.868 0.808 0.824 0.807]


## 8. Leakage, demonstrated

In [11]:
leaky = students[["study_hours", "attendance_pct", "prior_grade", "sleep_hours", "final_score"]].fillna(0)
Xl = leaky.to_numpy()
Xtr, Xte, ytr, yte = train_test_split(Xl, y_pass, 0.25, seed=0, stratify=y_pass)
s = StandardScaler().fit(Xtr)
m = LogisticRegressionGD(lr=0.3, epochs=400).fit(s.transform(Xtr), ytr)
print("with final_score, test accuracy:", round(m.score(s.transform(Xte), yte), 3))

honest = leaky.drop(columns=["final_score"]).to_numpy()
Xtr, Xte, ytr, yte = train_test_split(honest, y_pass, 0.25, seed=0, stratify=y_pass)
s = StandardScaler().fit(Xtr)
m = LogisticRegressionGD(lr=0.3, epochs=400).fit(s.transform(Xtr), ytr)
print("without, test accuracy:", round(m.score(s.transform(Xte), yte), 3))

with final_score, test accuracy: 0.993
without, test accuracy: 0.827


The 1.000 is a bug report, not a result. Any time a model is perfect on a noisy
social-science dataset, look for the column that encodes the answer.

## 9. Reflect

1. Which preparation step changed the result most in this notebook?
2. Why does the scaler's `transform` on the test set use the train means?
3. Name one preprocessing step you would still add for a production pipeline, and where it
   belongs.

Next: **NB-06-02** for regression and classification models.